# Session 0b - Python for Data Science

**Asynchronous · fast path ~1h · full path ~5h**

---

## Two paths through this notebook

This course is graduate-level in *data science*, not in programming. You need enough
Python to express an analysis and to debug it yourself - no more.

| | who | what to do |
|---|---|---|
| **Fast path** | you have written Python before | do only the **Checkpoint** cells (there are six). If all six are easy, you are done. |
| **Full path** | you have not, or the checkpoints were not easy | work through every section. Budget 5 hours across a few sittings. |

The six checkpoints map onto the six things the rest of the course actually uses. If
a checkpoint defeats you, that section is worth your time and the others may not be.

## What is deliberately not here

Classes, decorators, generators, `async`, packaging, type systems, testing
frameworks. All useful; none needed to complete this course. If you already know
them, good - you will not need them either.

## Learning objectives

1. Use the four data types this course relies on: numbers, strings, lists, dicts.
2. Write a comprehension, and say when a loop is clearer.
3. Write a function with a default argument and a docstring.
4. **Read a traceback and identify the line that actually failed.**
5. Recognise the difference between mutating an object and copying it.

In [ ]:
# Nothing to install. This notebook uses only the standard library and numpy.
import numpy as np

CHECKPOINTS = {}


def checkpoint(number, condition, message):
    """Record a checkpoint result so the summary at the end can report them."""
    CHECKPOINTS[number] = bool(condition)
    print(f"  checkpoint {number}: {'PASS' if condition else 'FAIL'}  {message}")


print("ready")

## §1 - Values and names

A name is a label attached to a value. `=` attaches it; it does not copy anything and
it does not declare a type.

In [ ]:
listings = 15293                  # int
median_price = 177.67             # float
city = "Barcelona"                # str
is_licensed = True                # bool
missing = None                    # the absence of a value

for name, value in [("listings", listings), ("median_price", median_price),
                    ("city", city), ("is_licensed", is_licensed), ("missing", missing)]:
    print(f"  {name:14s} = {str(value):12s} {type(value).__name__}")

### The three arithmetic operators that trip people up

| | | |
|---|---|---|
| `/` | true division, **always a float** | `7 / 2` → `3.5` |
| `//` | floor division | `7 // 2` → `3` |
| `%` | remainder | `7 % 2` → `1` |

`//` matters because indices must be integers, and `%` matters because `n % 2 == 0`
is how you test for even.

### Python will not convert types for you

In [ ]:
count = "42"                      # a string that looks like a number
try:
    print(count + 8)
except TypeError as exc:
    print(f"  '42' + 8 raises TypeError: {exc}")
print(f"  int('42') + 8 = {int(count) + 8}   <- convert explicitly")

This is why `price` in our dataset - stored as `'$409.00'` - cannot be averaged until
you parse it. Session 2 spends real time on that.

### f-strings

The only string formatting you need:

In [ ]:
n, rate = 15293, 0.6712
print(f"  {n} listings, {rate:.1%} licensed")
print(f"  padded: |{n:>10,}|  and rounded: |{rate:.3f}|")

`:,` adds thousands separators, `:.3f` fixes decimals, `:.1%` formats as a
percentage, `:>10` right-aligns in ten characters. You will use all four to make
printed output readable.

### Checkpoint 1

In [ ]:
# TODO Checkpoint 1. Set `answer` to the number of whole 40-listing batches in 15,293
# listings, plus the number left over. One expression, using // and %.
#
# For 15,293 that is 382 batches with 13 left over, so answer = 382 + 13 = 395.

answer = ...

## §2 - Lists

An ordered, mutable sequence. Indexing starts at **0**; negative indices count from
the end.

In [ ]:
districts = ["Eixample", "Ciutat Vella", "Gracia", "Sants", "Horta"]

print(f"  districts[0]   = {districts[0]}")
print(f"  districts[-1]  = {districts[-1]}      <- last, without needing len()")
print(f"  districts[1:3] = {districts[1:3]}   <- start included, stop EXCLUDED")
print(f"  districts[:2]  = {districts[:2]}")
print(f"  districts[::-1]= {districts[::-1]}")
print(f"  len(districts) = {len(districts)}")

The stop index being excluded is the single most common off-by-one source in Python.
`[1:3]` gives you two elements, not three. The upside: `a[:n]` and `a[n:]` together
give you exactly the original.

### Mutating versus copying - the important bit

In [ ]:
original = [3, 1, 2]

sorted_copy = sorted(original)          # returns a NEW list
print(f"  sorted(original) = {sorted_copy}   original still {original}")

reversed_copy = original[::-1]          # slicing returns a NEW list
print(f"  original[::-1]   = {reversed_copy}   original still {original}")

result = original.sort()                # sorts IN PLACE and returns None
print(f"  original.sort() returned {result} and original is now {original}")

> **`.sort()` returns `None`.** So does `.reverse()`, `.append()` and every other
> in-place method. Writing `x = my_list.sort()` gives you `None`, and it is one of the
> most common beginner bugs in Python.

The rule worth memorising: **a function that returns a new thing versus a method that
changes the existing thing.** `sorted(x)` / `x.sort()`. `x[::-1]` / `x.reverse()`.
This distinction reappears in pandas as `df.dropna()` versus `df.dropna(inplace=True)`
and it causes exactly the same confusion there.

### Checkpoint 2

In [ ]:
# TODO Checkpoint 2. `prices` must be left UNCHANGED. Set `top_two` to the two largest
# values, in descending order.

prices = [80, 175, 120, 240, 95]
top_two = ...

## §3 - Dictionaries

A mapping from keys to values. This is how you will hold configuration, results, and
anything keyed by a name.

In [ ]:
median_by_room = {"Entire home/apt": 231.0, "Private room": 62.0, "Shared room": 40.0}

print(f"  keys   : {list(median_by_room)}")
print(f"  lookup : {median_by_room['Private room']}")
print(f"  missing: {median_by_room.get('Hotel room', 'not recorded')}   <- .get with a default")

try:
    median_by_room["Hotel room"]
except KeyError as exc:
    print(f"  direct lookup of a missing key raises KeyError: {exc}")

`.get(key, default)` is the one to reach for whenever a key might be absent. Real
data always has absent keys.

### Iterating properly

In [ ]:
print("  iterating the dict gives KEYS only:")
for key in median_by_room:
    print(f"    {key}")

print("\n  .items() gives both:")
for room, price in median_by_room.items():
    print(f"    {room:16s} EUR {price:6.1f}")

### Checkpoint 3

In [ ]:
# TODO Checkpoint 3. Build `expensive`: a list of the room types whose median price is
# above 100. Then set `shared` to the price of "Shared room" or 0.0 if absent, without
# using try/except or an if statement.

expensive = ...
shared = ...

## §4 - Control flow

### Conditions

Note `elif`, and that indentation defines the block - there are no braces.

In [ ]:
def price_band(price):
    if price < 60:
        return "budget"
    elif price < 200:
        return "mid"
    else:
        return "premium"


print("  ", [price_band(p) for p in (40, 120, 350)])

### Truthiness

Empty containers, `0`, `""` and `None` are all **falsy**. This makes guards concise:

In [ ]:
for value in ([], [1], "", "x", 0, 1, None):
    print(f"  bool({value!r:6}) = {bool(value)}")

print("\n  so the idiomatic empty-check is `if not values:` rather than `if len(values) == 0:`")

### Loops

`for` over a sequence. `range(n)` for a count. `break` to stop early.

In [ ]:
prices = [80, 175, 120, 240, 95]

total = 0
for price in prices:
    total += price
print(f"  total = {total}")

first_premium = None
for price in prices:
    if price > 200:
        first_premium = price
        break                                   # stop at the first match
print(f"  first above 200 = {first_premium}")

print(f"  enumerate gives index and value:")
for i, price in enumerate(prices[:3]):
    print(f"    position {i} -> {price}")

### Checkpoint 4

In [ ]:
# TODO Checkpoint 4. Set `cleaned` to a list the SAME LENGTH as `raw`, where every
# negative value has been replaced by 0 and everything else left alone.
#
# Careful: this is a transformation, not a filter. The output must have 6 elements.

raw = [80, -1, 120, -1, 240, 95]
cleaned = ...

> **That checkpoint separates two things students conflate.** A comprehension with
> `if` at the *end* **filters** and changes the length. A conditional expression
> (`a if cond else b`) at the *front* **transforms** and preserves it.
>
> ```python
> [v for v in raw if v >= 0]           # 4 elements - FILTERED
> [v if v >= 0 else 0 for v in raw]    # 6 elements - TRANSFORMED
> ```
>
> Getting this wrong in pandas silently changes your row count, which changes every
> subsequent number, and nothing raises.

## §5 - Functions

A named, reusable block. Default arguments, a docstring, and a `return`.

In [ ]:
def summarise(values, decimals=2):
    """Return (count, mean) for a list of numbers, tolerating an empty list.

    The docstring is not decoration: it is what you will read in six weeks when you
    have forgotten what this does.
    """
    if not values:                          # guard the empty case BEFORE dividing
        return 0, 0.0
    return len(values), round(sum(values) / len(values), decimals)


print(f"  summarise([80, 120, 240]) = {summarise([80, 120, 240])}")
print(f"  summarise([], )           = {summarise([])}          <- no crash")
print(f"  decimals as a keyword     = {summarise([80, 120, 240], decimals=0)}")

Two habits from that example:

**Guard before you divide.** `sum([])/len([])` raises `ZeroDivisionError`. Real data
produces empty groups constantly.

**Pass keyword arguments by name.** `summarise(values, decimals=0)` is readable in six
weeks; `summarise(values, 0)` is not.

### One trap worth meeting now

A mutable default argument is created **once**, when the function is defined - not
each time it is called.

In [ ]:
def broken(item, collected=[]):             # DO NOT DO THIS
    collected.append(item)
    return collected


print(f"  broken('a') = {broken('a')}")
print(f"  broken('b') = {broken('b')}   <- 'a' is still there")


def fixed(item, collected=None):
    collected = [] if collected is None else collected
    collected.append(item)
    return collected


print(f"  fixed('a')  = {fixed('a')}")
print(f"  fixed('b')  = {fixed('b')}   <- correct")

### Checkpoint 5

In [ ]:
# TODO Checkpoint 5. Write `share_above(values, threshold=100)` returning the FRACTION
# of values strictly above the threshold, and 0.0 for an empty list.

def share_above(values, threshold=100):
    ...

## §6 - Reading a traceback

**This is the most important section in the notebook.**

Every later session assumes you can look at an error and know what to fix. Students
who can do this unblock themselves in a minute; students who cannot lose an hour
waiting for help.

In [ ]:
def load_prices(records):
    return [record["price"] for record in records]


def average_price(records):
    prices = load_prices(records)
    return sum(prices) / len(prices)


records = [{"price": 80}, {"price": 120}, {"cost": 240}]   # note the third key

try:
    average_price(records)
except KeyError:
    import traceback
    traceback.print_exc()

### How to read that

**Read it from the bottom.**

1. **The last line is what went wrong**: `KeyError: 'price'`. A dictionary was asked
   for a key it does not have.
2. **The last `File ... line N` block above it is where it went wrong** - inside
   `load_prices`, at the comprehension. That is the line to look at.
3. **The blocks above that are how you got there** - `average_price` called
   `load_prices`. Useful context, not the site of the bug.

The commonest mistake is reading the **top** and concluding the bug is in
`average_price`. It is not; that line is merely the call that led there.

### The five errors you will actually meet

| error | means | usual cause |
|---|---|---|
| `KeyError: 'x'` | no such dict key / DataFrame column | a typo, or the column really is absent |
| `TypeError: unsupported operand ...` | wrong types for the operation | a string that should be a number |
| `ValueError: could not convert ...` | right type, impossible value | `float('$409.00')` |
| `AttributeError: 'NoneType' has no ...` | you are using the result of something that returned `None` | `x = list.sort()` |
| `IndexError: list index out of range` | index past the end | an off-by-one, or an empty list |

### Checkpoint 6

In [ ]:
# TODO Checkpoint 6. Below is a traceback. Answer three things.
#
#     Traceback (most recent call last):
#       File "analysis.py", line 41, in <module>
#         report = summarise_city(rows)
#       File "analysis.py", line 22, in summarise_city
#         return build_table(cleaned)
#       File "analysis.py", line 9, in build_table
#         return total / count
#     ZeroDivisionError: division by zero
#
# error_type  = ...      # the exception name, as a string
# failing_line = ...     # the line number where it actually broke, as an int
# likely_cause = ...     # one short string: what is probably true of `count`

## §7 - The mutation trap, once more

This one deserves its own section because it is the source of a whole class of pandas
bugs you will hit in Session 2.

Two names can refer to **the same object**. Changing it through one name changes what
the other sees.

In [ ]:
a = [1, 2, 3]
b = a                     # NOT a copy - the same list, two names
b.append(4)
print(f"  a = {a}   <- a changed too, because b IS a")

c = [1, 2, 3]
d = c.copy()              # a genuine copy
d.append(4)
print(f"  c = {c}   d = {d}   <- independent")

In pandas this appears as a function that behaves differently the second time you run
it, because the first call modified the DataFrame you passed in. The defence is the
one used throughout this course:

```python
def clean(df):
    out = df.copy()        # never modify the caller's data
    ...
    return out
```

## §8 - Modules

You will only ever need these forms:

In [ ]:
import numpy as np                     # the whole module, renamed
from pathlib import Path               # one name out of a module

import sys
sys.path.append("..")
from src.data import load_raw, set_seed  # several names

print(f"  np.mean([1, 2, 3])   = {np.mean([1, 2, 3])}")
print(f"  Path('data') / 'raw' = {Path('data') / 'raw'}")
print("  from X import * is never used in this course - it hides where names came from")

## Checkpoint summary